# THYAO Hisse Senedi Analizi — Veri Bilimi Donem Projesi

**Ogrenci:** Ahmet Melih Cakalgil
**Tema:** Finans & Ekonomi — BIST (THYAO.IS)
**Veri kaynagi:** Yahoo Finance (`yfinance`)

## Arastirma Sorulari

1. THYAO hissesinin son 5 yildaki gunluk getiri dagilimi nasildir ve volatilite
   zaman icinde nasil degismistir?
2. Islem hacmi ile fiyat volatilitesi arasinda bir iliski var midir? Hareketli
   ortalama kesismeleri (golden cross / death cross) anlamli sinyaller uretiyor mu?
3. Teknik gostergeler kullanilarak ertesi gunun yonu (artis/azalis) siniflandirilabilir mi?


## 1. Kutuphanelerin Yuklenmesi

In [ ]:
# Veri cekme ve islem icin temel kutuphaneler
import yfinance as yf
import pandas as pd
import numpy as np

# Gorsellestirme
import matplotlib.pyplot as plt
import seaborn as sns

# Modelleme
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Grafik stili - tum notebook boyunca tutarli gorunum icin
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

print("Kutuphaneler basariyla yuklendi.")


## 2. Veri Toplama

THYAO.IS sembolu icin Yahoo Finance'ten son 5 yillik gunluk (daily) fiyat ve
hacim verisi cekilir. `auto_adjust=True` kullanilarak temettu/bolunme
duzeltmeleri otomatik uygulanir, boylece fiyat serisinde yapay siciramalar
olmaz.

In [ ]:
# THYAO.IS son 5 yillik gunluk veri
ticker = "THYAO.IS"
df_raw = yf.download(ticker, period="5y", interval="1d", auto_adjust=True)

print(f"Indirilen satir sayisi: {len(df_raw)}")
print(f"Tarih araligi: {df_raw.index.min().date()} -> {df_raw.index.max().date()}")
df_raw.head()


In [ ]:
# Ham veriyi kaydet (tekrar indirmeye gerek kalmasin diye)
df_raw.to_csv("../data/thyao_raw.csv")
df_raw.tail()


## 3. Veri Temizleme

Bu bolumde uc sey kontrol edilir:
1. **Eksik veri** — borsa tatil gunlerinde veri olmaz, bu normal; ama sutun
   icinde NaN var mi bakilir.
2. **Tip donusumu** — sutunlarin sayisal (float) tipte oldugundan emin olunur.
3. **Aykiri deger** — gunluk getiri uzerinden, istatistiksel olarak asiri
   uc degerler (örn. %20'den fazla tek gunluk hareket gibi gercek olmayan
   degerler) kontrol edilir. Not: gercek piyasa olaylarindan kaynaklanan
   buyuk hareketler (örn. kriz gunleri) silinmez, sadece veri hatasi olabilecek
   imkansiz degerler aranir.

In [ ]:
# yfinance multi-index sutun donduruyorsa duzlestir
if isinstance(df_raw.columns, pd.MultiIndex):
    df_raw.columns = df_raw.columns.get_level_values(0)

df = df_raw.copy()

# Eksik veri kontrolu
print("Eksik deger sayisi (sutun bazinda):")
print(df.isna().sum())


In [ ]:
# Eksik degerler varsa: once ileri doldurma (bir onceki gunun degeri ile),
# cunku fiyat serilerinde rastgele silme/ortalama ile doldurma yaniltici olur
if df.isna().sum().sum() > 0:
    df = df.ffill()
    print("Eksik degerler ileri doldurma (forward fill) ile dolduruldu.")
else:
    print("Eksik deger bulunmadi.")

# Tip donusumu kontrolu
print("\nSutun tipleri:")
print(df.dtypes)


In [ ]:
# Gunluk getiri hesapla (aykiri deger kontrolu icin gerekli)
df["Gunluk_Getiri"] = df["Close"].pct_change()

# Istatistiksel ozet
print(df["Gunluk_Getiri"].describe())

# Olasi veri hatasi olabilecek asiri degerleri tespit et (+-%50 ustu gunluk
# hareket BIST icin teorik olarak tavan/taban limitleri asar, gercek olamaz)
aykiri = df[df["Gunluk_Getiri"].abs() > 0.5]
print(f"\nSupheli (>%50 gunluk hareket) satir sayisi: {len(aykiri)}")
aykiri


In [ ]:
# Aykiri deger bulunursa cikar (BIST gunluk tavan/taban genelde %10 civarinda,
# %50 ustu bir hareket veri hatasidir, gercek piyasa hareketi degildir)
df = df[df["Gunluk_Getiri"].abs() <= 0.5].copy()

# Ilk satirdaki NaN getiriyi temizle (pct_change ilk satirda NaN doner)
df = df.dropna(subset=["Gunluk_Getiri"])

print(f"Temizleme sonrasi satir sayisi: {len(df)}")
df.describe()


## 4. Teknik Gosterge ve Feature Uretimi

Modelleme ve EDA icin kullanilacak teknik gostergeler hesaplanir:

- **SMA (Simple Moving Average):** 20 ve 50 gunluk hareketli ortalamalar —
  kisa ve uzun vadeli trend yonunu gosterir.
- **RSI (Relative Strength Index):** 14 gunluk, hissenin "asiri alim/asiri
  satim" bolgesinde olup olmadigini gosteren 0-100 araliginda bir gosterge.
- **Hacim degisimi:** gunluk islem hacminin onceki gune gore yuzde degisimi.
- **Volatilite:** 20 gunluk hareketli standart sapma (getiri uzerinden).

In [ ]:
# Hareketli ortalamalar (trend gostergeleri)
df["SMA_20"] = df["Close"].rolling(window=20).mean()
df["SMA_50"] = df["Close"].rolling(window=50).mean()

# RSI (Relative Strength Index) hesaplama - 14 gunluk standart pencere
delta = df["Close"].diff()
kazanc = delta.where(delta > 0, 0.0)
kayip = -delta.where(delta < 0, 0.0)

ortalama_kazanc = kazanc.rolling(window=14).mean()
ortalama_kayip = kayip.rolling(window=14).mean()

rs = ortalama_kazanc / ortalama_kayip
df["RSI_14"] = 100 - (100 / (1 + rs))

# Hacim degisimi (yuzde)
df["Hacim_Degisim"] = df["Volume"].pct_change()

# 20 gunluk volatilite (gunluk getirinin hareketli standart sapmasi)
df["Volatilite_20"] = df["Gunluk_Getiri"].rolling(window=20).std()

df[["Close", "SMA_20", "SMA_50", "RSI_14", "Hacim_Degisim", "Volatilite_20"]].tail()


In [ ]:
# Hareketli ortalama hesaplarindan kaynaklanan baslangic NaN satirlarini cikar
# (ilk 50 gun SMA_50 hesaplanamaz, bu beklenen bir durumdur)
df_model = df.dropna().copy()
print(f"Feature uretimi sonrasi kullanilabilir satir sayisi: {len(df_model)}")

# Temizlenmis veriyi kaydet
df_model.to_csv("../data/thyao_temiz.csv")


## 5. Kesifsel Veri Analizi (EDA)

Asagida 4 farkli gorsellestirme turu kullanilmistir:
1. Zaman serisi cizgi grafigi (fiyat + hareketli ortalamalar)
2. Histogram (gunluk getiri dagilimi)
3. Isi haritasi / korelasyon matrisi (feature'lar arasi iliski)
4. Saçilim grafigi (hacim degisimi vs volatilite)

In [ ]:
# 1) Zaman serisi: fiyat + hareketli ortalamalar
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_model.index, df_model["Close"], label="Kapanis fiyati", linewidth=1)
ax.plot(df_model.index, df_model["SMA_20"], label="SMA 20", linewidth=1)
ax.plot(df_model.index, df_model["SMA_50"], label="SMA 50", linewidth=1)
ax.set_title("THYAO Kapanis Fiyati ve Hareketli Ortalamalar (Son 5 Yil)")
ax.set_xlabel("Tarih")
ax.set_ylabel("Fiyat (TL)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 2) Histogram: gunluk getiri dagilimi
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df_model["Gunluk_Getiri"], bins=80, kde=True, ax=ax)
ax.set_title("THYAO Gunluk Getiri Dagilimi")
ax.set_xlabel("Gunluk Getiri")
ax.set_ylabel("Frekans")
ax.axvline(0, color="red", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

print(f"Ortalama gunluk getiri: {df_model['Gunluk_Getiri'].mean():.5f}")
print(f"Standart sapma: {df_model['Gunluk_Getiri'].std():.5f}")
print(f"Carpiklik (skewness): {df_model['Gunluk_Getiri'].skew():.3f}")
print(f"Kurtosis: {df_model['Gunluk_Getiri'].kurt():.3f}")


In [ ]:
# 3) Korelasyon isi haritasi
feature_cols = ["Close", "Volume", "Gunluk_Getiri", "SMA_20", "SMA_50",
                 "RSI_14", "Hacim_Degisim", "Volatilite_20"]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df_model[feature_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax)
ax.set_title("Feature'lar Arasi Korelasyon Matrisi")
plt.tight_layout()
plt.show()


In [ ]:
# 4) Sacilim grafigi: hacim degisimi vs volatilite
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df_model["Hacim_Degisim"], df_model["Volatilite_20"], alpha=0.4, s=15)
ax.set_title("Hacim Degisimi ile 20-Gunluk Volatilite Iliskisi")
ax.set_xlabel("Gunluk Hacim Degisimi (%)")
ax.set_ylabel("20 Gunluk Volatilite")
plt.tight_layout()
plt.show()

korelasyon = df_model["Hacim_Degisim"].corr(df_model["Volatilite_20"])
print(f"Hacim degisimi - Volatilite korelasyonu: {korelasyon:.3f}")


## 6. Arastirma Sorusu 1: Getiri Dagilimi ve Volatilite Zaman Icinde Nasil Degisti?

Yillik bazda volatilite hesaplanarak hangi donemlerde THYAO'nun daha
oynak (volatil) oldugu incelenir.

In [ ]:
# Yillik volatilite (gunluk getirinin yillik standart sapmasi, sqrt(252) ile yillik hale getirilir)
df_model["Yil"] = df_model.index.year
yillik_volatilite = df_model.groupby("Yil")["Gunluk_Getiri"].std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(10, 5))
yillik_volatilite.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("THYAO Yillik (Annualize) Volatilite")
ax.set_xlabel("Yil")
ax.set_ylabel("Yillik Volatilite")
plt.tight_layout()
plt.show()

print(yillik_volatilite)


**Yorum:** [BURAYA grafikteki sonucu THYAO baglaminda yorumla — örn.
hangi yil en yuksek volatilite gorulmus, bunun sektorel/ekonomik bir nedeni
olabilir mi (doviz kuru, yakit fiyatlari, faiz kararlari gibi). Bu yorumu
kendi gozlemine gore yazmalisin, ben sadece grafigi urettim.]

## 7. Arastirma Sorusu 2: Hacim-Volatilite Iliskisi ve Hareketli Ortalama Kesismeleri

Golden cross (SMA_20'nin SMA_50'yi yukari kesmesi) ve death cross (asagi
kesmesi) sinyalleri tespit edilip, bu sinyal sonrasi N gunluk getiri
incelenir.

In [ ]:
# Golden cross / death cross tespiti
df_model["Sinyal"] = np.where(df_model["SMA_20"] > df_model["SMA_50"], 1, -1)
df_model["Sinyal_Degisimi"] = df_model["Sinyal"].diff()

golden_cross_tarihleri = df_model[df_model["Sinyal_Degisimi"] == 2].index
death_cross_tarihleri = df_model[df_model["Sinyal_Degisimi"] == -2].index

print(f"Golden cross sayisi: {len(golden_cross_tarihleri)}")
print(f"Death cross sayisi: {len(death_cross_tarihleri)}")


In [ ]:
# Her kesisme sonrasi 10 gunluk getiriyi hesapla
def n_gun_sonrasi_getiri(tarihler, df, n=10):
    sonuclar = []
    for tarih in tarihler:
        idx = df.index.get_loc(tarih)
        if idx + n < len(df):
            baslangic_fiyat = df["Close"].iloc[idx]
            bitis_fiyat = df["Close"].iloc[idx + n]
            getiri = (bitis_fiyat - baslangic_fiyat) / baslangic_fiyat
            sonuclar.append(getiri)
    return sonuclar

golden_getiriler = n_gun_sonrasi_getiri(golden_cross_tarihleri, df_model)
death_getiriler = n_gun_sonrasi_getiri(death_cross_tarihleri, df_model)

print(f"Golden cross sonrasi 10 gunluk ortalama getiri: {np.mean(golden_getiriler):.4f}" if golden_getiriler else "Golden cross verisi yetersiz")
print(f"Death cross sonrasi 10 gunluk ortalama getiri: {np.mean(death_getiriler):.4f}" if death_getiriler else "Death cross verisi yetersiz")


**Yorum:** [BURAYA golden/death cross sonuclarini yorumla — eger
golden cross sonrasi pozitif, death cross sonrasi negatif getiri ortalamasi
varsa bu sinyalin bir isaret tasidigini soyleyebilirsin; ama ornek sayisi
azsa (örn. <10 kesisme) bu sonucun istatistiksel olarak guclu olmadigini da
belirtmen gerekir.]

## 8. Arastirma Sorusu 3: Siniflandirma Modeli — Ertesi Gun Yon Tahmini

**Hedef degisken:** Ertesi gunun kapanis fiyati bugunkune gore yukseldi mi (1)
dustu mu (0)?

**Onemli metodolojik nokta:** Zaman serisi verisinde train/test ayrimi
**rastgele degil, kronolojik** olarak yapilmalidir. Aksi halde model gelecekteki
veriyi "gorerek" ogrenir (data leakage) ve gercekte olmayan bir basari
gosterir.

In [ ]:
# Hedef degisken: ertesi gun fiyat yukseldi mi? (1 = evet, 0 = hayir)
df_model["Hedef"] = (df_model["Close"].shift(-1) > df_model["Close"]).astype(int)

# Son satirda ertesi gun verisi olmadigi icin NaN/gecersiz olur, cikar
df_ml = df_model.dropna(subset=["Hedef"]).copy()

# Kullanilacak feature'lar (bugune kadar bilinen bilgiler - gelecek bilgisi yok)
features = ["RSI_14", "SMA_20", "SMA_50", "Hacim_Degisim", "Volatilite_20", "Gunluk_Getiri"]
X = df_ml[features]
y = df_ml["Hedef"]

print(f"Toplam ornek sayisi: {len(X)}")
print(f"Sinif dagilimi:\n{y.value_counts(normalize=True)}")


In [ ]:
# Kronolojik train/test ayrimi (ilk %80 train, son %20 test)
# shuffle=False kritik: zaman serisinde rastgele karistirma data leakage'e yol acar
split_index = int(len(X) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Egitim seti: {len(X_train)} ornek ({X_train.index.min().date()} -> {X_train.index.max().date()})")
print(f"Test seti: {len(X_test)} ornek ({X_test.index.min().date()} -> {X_test.index.max().date()})")


In [ ]:
# Model: Random Forest siniflandirici
# Random Forest secildi cunku: a) ozellik olceklendirmesine ihtiyac duymaz,
# b) dogrusal olmayan iliskileri yakalayabilir, c) overfitting'e lojistik
# regresyona kiyasla biraz daha dayaniklidir (cok derin olmayan agaclarla)
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Test seti basari orani (accuracy):", accuracy_score(y_test, y_pred))
print("\nSiniflandirma raporu:\n", classification_report(y_test, y_pred))


In [ ]:
# Baseline karsilastirma: "her zaman cogunluk sinifini tahmin et" stratejisi
# Modelin gercekten bir sey ogrenip ogrenmedigini anlamak icin bu karsilastirma sarttir
cogunluk_sinif = y_train.mode()[0]
baseline_tahmin = np.full(len(y_test), cogunluk_sinif)
baseline_accuracy = accuracy_score(y_test, baseline_tahmin)

print(f"Baseline (cogunluk sinif) basari orani: {baseline_accuracy:.4f}")
print(f"Model basari orani: {accuracy_score(y_test, y_pred):.4f}")
print(f"Fark: {accuracy_score(y_test, y_pred) - baseline_accuracy:+.4f}")


In [ ]:
# Karisiklik matrisi (confusion matrix) gorsellestirmesi
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Dusus", "Yukselis"], yticklabels=["Dusus", "Yukselis"], ax=ax)
ax.set_xlabel("Tahmin Edilen")
ax.set_ylabel("Gercek")
ax.set_title("Karisiklik Matrisi - Test Seti")
plt.tight_layout()
plt.show()


In [ ]:
# Feature onem siralamasi - hangi gostergeler modelin kararinda daha etkili
onem = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
onem.plot(kind="barh", ax=ax, color="darkorange")
ax.set_title("Feature Onem Siralamasi (Random Forest)")
ax.set_xlabel("Onem Skoru")
plt.tight_layout()
plt.show()

print(onem)


**Yorum:** [BURAYA model basarisini baseline ile karsilastirarak yorumla.
Eger model accuracy'si baseline'a cok yakinsa (örn. %50-55 civarinda), bu
THYAO'nun gunluk yon tahmininin teknik gostergelerle kolay yakalanamadigini,
piyasanin kisa vadede nispeten "etkin" davrandigini gosterir — bu beklenen
ve savunulabilir bir sonuctur, basarisizlik degildir. Eger model baseline'i
acikca geciyorsa, bunun hangi feature'dan (yukaridaki onem siralamasina bakarak)
kaynaklandigini tartis.]

## 9. Genel Sonuc ve Sinirlamalar

**Bulgular Ozeti:**
- [BURAYA arastirma sorusu 1, 2, 3 icin elde ettigin somut bulgulari 2-3
  cumleyle ozetle]

**Sinirlamalar:**
- Analiz tek bir hisseye (THYAO) odaklanmistir, bulgular diger BIST hisselerine
  genellenemez.
- Kisa vadeli fiyat yonu tahmini, finansal piyasalarin yari-etkin yapisi
  nedeniyle dogasi geregi zordur; model sonuclari bu cerceve icinde
  degerlendirilmelidir.
- Kullanilan model kapsamli hiperparametre optimizasyonundan gecmemistir
  (proje gereksinimleri dogrultusunda tek savunulabilir model yeterli
  gorulmustur).
- Islem maliyetleri (komisyon, vergi, kayma/slippage) modele dahil
  edilmemistir; bu nedenle model bir "yatirim stratejisi" olarak degil,
  yalniz analitik bir alistirma olarak degerlendirilmelidir.

**Ogrenilenler:**
- [BURAYA bu proje surecinde sen ne ogrendigini yaz — örn. veri sizintisi
  (data leakage) konusunda nelere dikkat ettigini, zaman serisi train/test
  ayriminin neden farkli oldugunu, AI ile calisirken hangi kararlari
  sorguladigini]
